# Evaluate multiple checkpoints on PathVQA val and test

Notebook này dành cho Kaggle. Hãy bật GPU, gắn PathVQA (thư mục có `train.json`, `val.json`, `test.json`, `images/`) và các checkpoint. Mặc định mỗi checkpoint được đánh giá trên cả val và test; có thể sửa `EVAL_SPLITS` để chỉ chạy một split.

Do `PathVqaDataset` chỉ đọc file tên `test.json` khi evaluation, notebook tạo annotation tạm cho từng split mà không sửa dữ liệu gốc. Kết quả JSON/CSV được xuất riêng theo split và dưới dạng bảng gộp.

In [ ]:
from pathlib import Path

REPO_URL = 'https://github.com/fantastichaha11/SelTDA.git'
BRANCH = 'feat/pseudo-label-filter'
REPO_DIR = Path('/kaggle/working/SelTDA')

# Đặt đường dẫn cụ thể nếu đã biết; giữ None để notebook tự tìm.
PATHVQA_ROOT = None
PATHVQA_SEARCH_ROOTS = [Path('/kaggle/working/pathvqa'), Path('/kaggle/input')]
EVAL_SPLITS = ['val', 'test']

# Có thể truyền một hoặc nhiều checkpoint tại đây.
CHECKPOINT_PATHS = [
    # '/kaggle/input/my-checkpoints/checkpoint_09.pth',
    # '/kaggle/input/my-checkpoints/checkpoint_19.pth',
]
CHECKPOINT_SEARCH_ROOTS = [Path('/kaggle/input'), Path('/kaggle/working')]
CHECKPOINT_GLOB = 'checkpoint_*.pth'
MAX_CHECKPOINTS = None

OUTPUT_ROOT = Path('/kaggle/working/pathvqa_eval')
MAX_GPUS = 2
BATCH_SIZE_TEST = 4
K_TEST = 128
INFERENCE = 'rank'
RERUN_EXISTING = False

OUTPUT_ROOT.mkdir(parents=True, exist_ok=True)

In [ ]:
import subprocess
import sys

if not (REPO_DIR / '.git').exists():
    subprocess.run(
        ['git', 'clone', '--depth', '1', '--branch', BRANCH, REPO_URL, str(REPO_DIR)],
        check=True,
    )
else:
    print(f'Using existing repository: {REPO_DIR}')

packages = [
    'omegaconf==2.3.0',
    'hydra-core==1.3.2',
    'timm==0.4.12',
    'fairscale==0.4.13',
    'transformers==4.36.1',
    'pandas',
    'pyyaml',
]
subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', *packages], check=True)
print(f'Repository ready: {REPO_DIR}')

In [ ]:
import json

if not EVAL_SPLITS or len(EVAL_SPLITS) != len(set(EVAL_SPLITS)):
    raise ValueError('EVAL_SPLITS must be a non-empty list without duplicates.')
unsupported_splits = set(EVAL_SPLITS) - {'val', 'test'}
if unsupported_splits:
    raise ValueError(f'Unsupported PathVQA evaluation splits: {sorted(unsupported_splits)}')

def is_pathvqa_root(path):
    path = Path(path)
    required_files = ['train.json', *[f'{split_name}.json' for split_name in EVAL_SPLITS]]
    required_dirs = [path / 'images' / split_name for split_name in EVAL_SPLITS]
    return all((path / name).is_file() for name in required_files) and all(
        directory.is_dir() for directory in required_dirs
    )

def find_pathvqa_root(explicit_root, search_roots):
    if explicit_root is not None:
        candidate = Path(explicit_root).expanduser().resolve()
        if not is_pathvqa_root(candidate):
            raise FileNotFoundError(f'PATHVQA_ROOT is invalid: {candidate}')
        return candidate

    checked = set()
    for search_root in search_roots:
        search_root = Path(search_root)
        if not search_root.exists():
            continue
        direct_candidates = [search_root, search_root / 'pathvqa', search_root / 'PathVQA']
        for candidate in direct_candidates:
            resolved = candidate.resolve()
            if resolved not in checked and is_pathvqa_root(resolved):
                return resolved
            checked.add(resolved)
        for train_json in search_root.rglob('train.json'):
            candidate = train_json.parent.resolve()
            if candidate not in checked and is_pathvqa_root(candidate):
                return candidate
            checked.add(candidate)
    raise FileNotFoundError(
        'Could not find PathVQA. Attach the dataset or set PATHVQA_ROOT explicitly.'
    )

pathvqa_root = find_pathvqa_root(PATHVQA_ROOT, PATHVQA_SEARCH_ROOTS)
records_by_split = {}
for eval_split in EVAL_SPLITS:
    with (pathvqa_root / f'{eval_split}.json').open(encoding='utf-8') as file:
        records = json.load(file)
    if not records:
        raise ValueError(f'PathVQA {eval_split}.json is empty.')
    missing_images = []
    for record in records:
        image_path = record.get('image') or record.get('image_path')
        if not image_path:
            raise ValueError(f'Missing image field in {eval_split} record: {record}')
        if not (pathvqa_root / 'images' / image_path).is_file():
            missing_images.append(image_path)
            if len(missing_images) >= 10:
                break
    if missing_images:
        raise FileNotFoundError(
            f'Missing PathVQA {eval_split} images, examples: {missing_images}'
        )
    records_by_split[eval_split] = records

print(f'PathVQA root: {pathvqa_root}')
for eval_split, records in records_by_split.items():
    print(f'{eval_split}: {len(records):,} examples')

In [ ]:
import re

def checkpoint_label(path):
    raw = f'{path.parent.name}_{path.stem}'
    return re.sub(r'[^A-Za-z0-9_.-]+', '_', raw).strip('_.-') or 'checkpoint'

if CHECKPOINT_PATHS:
    discovered_checkpoints = [Path(path).expanduser().resolve() for path in CHECKPOINT_PATHS]
else:
    discovered_checkpoints = []
    for search_root in CHECKPOINT_SEARCH_ROOTS:
        search_root = Path(search_root)
        if search_root.exists():
            discovered_checkpoints.extend(search_root.rglob(CHECKPOINT_GLOB))
    discovered_checkpoints = sorted({path.resolve() for path in discovered_checkpoints})

valid_checkpoints = []
for checkpoint in discovered_checkpoints:
    if not checkpoint.is_file():
        raise FileNotFoundError(f'Checkpoint not found: {checkpoint}')
    if checkpoint.stat().st_size <= 1_000_000:
        print(f'Skipping suspiciously small checkpoint: {checkpoint}')
        continue
    valid_checkpoints.append(checkpoint)

if MAX_CHECKPOINTS is not None:
    valid_checkpoints = valid_checkpoints[:MAX_CHECKPOINTS]
if not valid_checkpoints:
    raise FileNotFoundError('No valid checkpoints found. Fill CHECKPOINT_PATHS or adjust the search roots.')

checkpoint_paths = {}
for checkpoint in valid_checkpoints:
    base_label = checkpoint_label(checkpoint)
    label = base_label
    suffix = 2
    while label in checkpoint_paths:
        label = f'{base_label}_{suffix}'
        suffix += 1
    checkpoint_paths[label] = checkpoint

print(f'Checkpoints to evaluate: {len(checkpoint_paths)}')
for label, checkpoint in checkpoint_paths.items():
    print(f'  {label}: {checkpoint}')

In [ ]:
import json
import shutil
import yaml

def answer_candidates(records):
    answers = set()
    for record in records:
        values = record.get('answer', [])
        if isinstance(values, str):
            values = [values]
        for value in values:
            normalized = str(value).strip()
            if normalized:
                answers.add(normalized)
    if not answers:
        raise ValueError('Could not build an answer list for evaluation.')
    return sorted(answers)

annotation_roots = {}
config_paths = {}
config_args = {}
for eval_split in EVAL_SPLITS:
    annotation_root = OUTPUT_ROOT / 'annotations' / eval_split
    annotation_root.mkdir(parents=True, exist_ok=True)
    shutil.copy2(pathvqa_root / 'train.json', annotation_root / 'train.json')
    shutil.copy2(pathvqa_root / f'{eval_split}.json', annotation_root / 'test.json')
    with (pathvqa_root / f'{eval_split}.json').open(encoding='utf-8') as file:
        split_records = json.load(file)
    with (annotation_root / 'answer_list.json').open('w', encoding='utf-8') as file:
        json.dump(answer_candidates(split_records), file, ensure_ascii=False)

    config_path = REPO_DIR / 'configs' / f'pathvqa_eval_{eval_split}_kaggle.yaml'
    config_path.parent.mkdir(parents=True, exist_ok=True)
    eval_config = {
        'ann_root': str(annotation_root),
        'vqa_root': str(pathvqa_root / 'images'),
        'train_files': ['train'],
        'dataset_name': 'pathvqa',
        'truncate_train_dataset_to': None,
        'pretrained': str(next(iter(checkpoint_paths.values()))),
        'vit': 'base',
        'batch_size_train': 1,
        'batch_size_test': BATCH_SIZE_TEST,
        'vit_grad_ckpt': False,
        'vit_ckpt_layer': 0,
        'init_lr': 2e-5,
        'image_size': 480,
        'k_test': K_TEST,
        'inference': INFERENCE,
        'weight_decay': 0.05,
        'min_lr': 0,
        'max_epoch': 1,
        'torch_home': '/kaggle/working/torch_home',
        'wandb': False,
        'save_last_only': True,
    }
    with config_path.open('w', encoding='utf-8') as file:
        yaml.safe_dump(eval_config, file, sort_keys=False)
    annotation_roots[eval_split] = annotation_root
    config_paths[eval_split] = config_path
    config_args[eval_split] = str(config_path.relative_to(REPO_DIR))
    print(f'[{eval_split}] Config: {config_path}')

In [ ]:
import torch

available_gpus = torch.cuda.device_count()
if available_gpus < 1:
    raise RuntimeError('No CUDA GPU detected. Enable a GPU accelerator in Kaggle settings.')
gpu_count = min(MAX_GPUS, available_gpus)
print(f'Using {gpu_count} of {available_gpus} available GPU(s)')
for index in range(available_gpus):
    print(f'  cuda:{index}: {torch.cuda.get_device_name(index)}')

In [ ]:
import os

result_files = {eval_split: {} for eval_split in EVAL_SPLITS}
run_environment = os.environ.copy()
run_environment['TOKENIZERS_PARALLELISM'] = 'false'
run_environment['TORCH_HOME'] = '/kaggle/working/torch_home'

for eval_split in EVAL_SPLITS:
    for label, checkpoint in checkpoint_paths.items():
        output_dir = OUTPUT_ROOT / eval_split / label
        result_file = output_dir / 'result/vqa_result.json'
        log_file = output_dir / 'inference.log'
        output_dir.mkdir(parents=True, exist_ok=True)

        if result_file.is_file() and not RERUN_EXISTING:
            print(f'[{eval_split}/{label}] Reusing result: {result_file}')
            result_files[eval_split][label] = result_file
            continue

        command = [
            sys.executable,
            '-m',
            'torch.distributed.run',
            '--standalone',
            f'--nproc_per_node={gpu_count}',
            'train_vqa.py',
            f'--config={config_args[eval_split]}',
            f'--output_dir={output_dir}',
            '--evaluate',
            '--no-resume',
            '--overrides',
            f'pretrained={checkpoint}',
        ]
        print(f'[{eval_split}/{label}] Evaluating {checkpoint}')
        with log_file.open('w', encoding='utf-8') as log:
            process = subprocess.run(
                command,
                cwd=REPO_DIR,
                env=run_environment,
                stdout=log,
                stderr=subprocess.STDOUT,
                text=True,
            )
        if process.returncode != 0:
            log_tail = log_file.read_text(encoding='utf-8', errors='replace').splitlines()[-80:]
            print('\n'.join(log_tail))
            raise RuntimeError(
                f'Inference failed for {eval_split}/{label}; see {log_file}'
            )
        if not result_file.is_file():
            raise FileNotFoundError(f'Inference result is missing: {result_file}')
        result_files[eval_split][label] = result_file
        print(f'[{eval_split}/{label}] Result: {result_file}')

In [ ]:
metrics_by_split = {eval_split: {} for eval_split in EVAL_SPLITS}
for eval_split in EVAL_SPLITS:
    annotation_file = pathvqa_root / f'{eval_split}.json'
    for label, result_file in result_files[eval_split].items():
        command = [
            sys.executable,
            'pathvqa_eval.py',
            str(result_file),
            '--annotation-file',
            str(annotation_file),
        ]
        process = subprocess.run(
            command,
            cwd=REPO_DIR,
            capture_output=True,
            text=True,
        )
        if process.stdout:
            print(f'[{eval_split}/{label}] {process.stdout.strip()}')
        if process.returncode != 0:
            print(process.stderr)
            raise RuntimeError(f'PathVQA scoring failed for {eval_split}/{label}')

        metrics_file = result_file.with_name('pathvqa_eval.json')
        if not metrics_file.is_file():
            raise FileNotFoundError(f'Metrics file is missing: {metrics_file}')
        with metrics_file.open(encoding='utf-8') as file:
            metrics_by_split[eval_split][label] = json.load(file)

metrics_by_split

In [ ]:
import pandas as pd

summary = {eval_split: {} for eval_split in EVAL_SPLITS}
rows = []
for eval_split in EVAL_SPLITS:
    split_rows = []
    for label, metrics in metrics_by_split[eval_split].items():
        entry = {
            'split': eval_split,
            'checkpoint': label,
            'checkpoint_path': str(checkpoint_paths[label]),
            'result_file': str(result_files[eval_split][label]),
            'metrics': metrics,
        }
        summary[eval_split][label] = entry
        row = {
            'split': eval_split,
            'checkpoint': label,
            'checkpoint_path': str(checkpoint_paths[label]),
            'result_file': str(result_files[eval_split][label]),
            **metrics,
        }
        rows.append(row)
        split_rows.append(row)

    split_json = OUTPUT_ROOT / f'pathvqa_{eval_split}_summary.json'
    split_csv = OUTPUT_ROOT / f'pathvqa_{eval_split}_summary.csv'
    with split_json.open('w', encoding='utf-8') as file:
        json.dump(summary[eval_split], file, indent=2, ensure_ascii=False)
    split_df = pd.DataFrame(split_rows)
    if 'overall' in split_df.columns:
        split_df = split_df.sort_values('overall', ascending=False)
    split_df.to_csv(split_csv, index=False)
    print(f'[{eval_split}] JSON summary: {split_json}')
    print(f'[{eval_split}] CSV summary:  {split_csv}')

summary_json = OUTPUT_ROOT / 'pathvqa_eval_summary.json'
summary_csv = OUTPUT_ROOT / 'pathvqa_eval_summary.csv'
with summary_json.open('w', encoding='utf-8') as file:
    json.dump(summary, file, indent=2, ensure_ascii=False)

summary_df = pd.DataFrame(rows)
if 'overall' in summary_df.columns:
    summary_df = summary_df.sort_values(['split', 'overall'], ascending=[True, False])
summary_df.to_csv(summary_csv, index=False)

display_df = summary_df.copy()
metric_columns = [column for column in display_df.columns if column not in {
    'split', 'checkpoint', 'checkpoint_path', 'result_file'
}]
for column in metric_columns:
    display_df[column] = display_df[column].map(
        lambda value: f'{100 * value:.2f}%' if isinstance(value, (int, float)) else value
    )
print(f'JSON summary: {summary_json}')
print(f'CSV summary:  {summary_csv}')
display(display_df)